In [0]:
%pip install xgboost scikit-learn pandas

In [0]:
import sys
import importlib
import pandas as pd
import numpy as np
SRC_PATH = "/Workspace/Users/ariamostajeran99@gmail.com/stock_project_V2/stock-mlops-databricks/src"

if SRC_PATH not in sys.path:
    sys.path.append(SRC_PATH)

import config
import model_trainer
import dataset_builder

importlib.reload(config)
importlib.reload(model_trainer)
importlib.reload(dataset_builder)

from universe import TRAIN_UNIVERSE
from config import (
    ML_DATASET_TABLE_NAME,
    MODEL_METRICS_TABLE_NAME,
    MODEL_PREDICTIONS_TABLE_NAME,
    FEATURE_IMPORTANCE_TABLE_NAME,
    BASELINE_COMPARE_TABLE_NAME,
    ROLLING_TRAIN_YEARS,
    ROLLING_TEST_MONTHS,
    TOP_K_FEATURES,
    META_THRESHOLD_GRID,
    META_MIN_TRADES
)
from model_trainer import ModelTrainer
from dataset_builder import DatasetBuilder

In [0]:
ml_df = spark.table(ML_DATASET_TABLE_NAME).toPandas()

print("ML dataset rows:", len(ml_df))
print("ML dataset cols:", len(ml_df.columns))
print("Tickers:", sorted(ml_df["Ticker"].unique()))
ml_df.head()

In [0]:
from config import TARGET_HORIZON_DAYS, TARGET_RETURN_THRESHOLD

builder = DatasetBuilder(
    target_horizon_days=TARGET_HORIZON_DAYS,
    target_return_threshold=TARGET_RETURN_THRESHOLD
)
global_feature_cols = builder.get_feature_columns(ml_df)
print("Global feature count:", len(global_feature_cols))
print(global_feature_cols[:20])

In [0]:
trainer = ModelTrainer()

tickers = [t for t in TRAIN_UNIVERSE if t in sorted(ml_df["Ticker"].unique())]

all_metrics = []
all_baseline_compare = []
all_predictions = []
all_feature_importance = []
all_probability_summary = []
all_feature_selection = []
all_threshold_optimization = []


In [0]:

for ticker in tickers:
    print(f"\nTraining model for {ticker} ...")

    ticker_feature_cols = builder.get_feature_columns(ml_df, ticker=ticker)

    ticker_df, model_feature_cols = trainer.prepare_single_ticker_dataset(
        ml_df=ml_df,
        feature_cols=ticker_feature_cols,
        ticker=ticker
    )

    ticker_df = ticker_df.dropna(
        subset=model_feature_cols + ["target_class", "naive_prediction_class"]
    ).reset_index(drop=True)

    print(f"{ticker} rows after ticker-specific dropna:", len(ticker_df))
    print(f"{ticker} feature count:", len(model_feature_cols))

    splits = trainer.get_walk_forward_splits(
        ticker_df,
        train_years=ROLLING_TRAIN_YEARS,
        test_months=ROLLING_TEST_MONTHS
    )

    if len(splits) == 0:
        print(f"Skipping {ticker}: no valid rolling splits")
        continue

    for split_id, (train_df, test_df) in enumerate(splits, start=1):
        X_train_full = train_df[model_feature_cols].copy()
        y_train = train_df["target_class"].copy()

        X_test_full = test_df[model_feature_cols].copy()
        y_test = test_df["target_class"].copy()

        # Pass 1: feature selection
        model_full = trainer.fit_xgboost(X_train_full, y_train)
        top_features = trainer.select_top_features(model_full, model_feature_cols, TOP_K_FEATURES)

        all_feature_selection.append({
            "Ticker": ticker,
            "split_id": split_id,
            "selected_feature_count": len(top_features),
            "selected_features": ", ".join(top_features)
        })

        # Pass 2: final base model with top features
        X_train = X_train_full[top_features].copy()
        X_test = X_test_full[top_features].copy()

        model = trainer.fit_xgboost(X_train, y_train)

        train_pred = model.predict(X_train)
        train_prob = model.predict_proba(X_train)

        test_pred = model.predict(X_test)
        test_prob = model.predict_proba(X_test)

        train_metrics = trainer.evaluate_predictions(y_train, train_pred)
        test_metrics = trainer.evaluate_predictions(y_test, test_pred)

        all_metrics.append({
            "Ticker": ticker,
            "model_name": f"p2_xgboost_{ticker}_v3",
            "split_id": split_id,
            "dataset_split": "train",
            **train_metrics
        })

        all_metrics.append({
            "Ticker": ticker,
            "model_name": f"p2_xgboost_{ticker}_v3",
            "split_id": split_id,
            "dataset_split": "test",
            **test_metrics
        })

        # ---- meta-labeling ----
        meta_train_df, meta_feature_cols = trainer.build_meta_dataset(
            train_df, train_pred, train_prob, top_features
        )
        meta_test_df, _ = trainer.build_meta_dataset(
            test_df, test_pred, test_prob, top_features
        )

        meta_model = trainer.fit_meta_model(meta_train_df, meta_feature_cols)

        if meta_model is None:
            meta_train_prob = meta_train_df["predicted_probability"].values
            meta_test_prob = meta_test_df["predicted_probability"].values
        else:
            eligible_train = meta_train_df[meta_train_df["predicted_class"] != 1].copy()
            meta_train_prob = np.zeros(len(meta_train_df))
            if len(eligible_train) > 0:
                probs = meta_model.predict_proba(eligible_train[meta_feature_cols])[:, 1]

                mask = (meta_train_df["predicted_class"] != 1).values
                meta_train_prob[mask] = probs

            eligible_test = meta_test_df[meta_test_df["predicted_class"] != 1].copy()
            meta_test_prob = np.zeros(len(meta_test_df))       
            if len(eligible_test) > 0:
                probs = meta_model.predict_proba(eligible_test[meta_feature_cols])[:, 1]

                mask = (meta_test_df["predicted_class"] != 1).values
                meta_test_prob[mask] = probs

        # ---- threshold optimization on train ----
        chosen_threshold, threshold_df = trainer.optimize_threshold(
            meta_train_df,
            meta_train_prob,
            threshold_grid=META_THRESHOLD_GRID,
            min_trades=META_MIN_TRADES
        )

        threshold_df["Ticker"] = ticker
        threshold_df["split_id"] = split_id
        all_threshold_optimization.append(threshold_df)

        # ---- apply meta filter to test ----
        final_test_df = trainer.apply_meta_filter(
            meta_test_df,
            meta_test_prob,
            chosen_threshold=chosen_threshold
        )

        final_test_df["is_correct"] = (
            final_test_df["predicted_class"] == final_test_df["target_class"]
        ).astype(int)

        prediction_output_df = trainer.build_prediction_output(
            final_test_df,
            model_name=f"p2_xgboost_{ticker}_v3",
            chosen_threshold=chosen_threshold
        )
        prediction_output_df["dataset_split"] = "test"
        prediction_output_df["split_id"] = split_id

        all_predictions.append(prediction_output_df)

        # baseline compare on selected trades only
        traded = final_test_df[final_test_df["selected_trade"] == 1].copy()

        if len(traded) > 0:
            naive_acc = (traded["naive_prediction_class"] == traded["target_class"]).mean()
            xgb_acc = (traded["predicted_class"] == traded["target_class"]).mean()
        else:
            naive_acc = np.nan
            xgb_acc = np.nan

        all_baseline_compare.append({
            "Ticker": ticker,
            "model_name": f"p2_xgboost_{ticker}_v3",
            "split_id": split_id,
            "chosen_threshold": chosen_threshold,
            "n_trades": len(traded),
            "naive_test_accuracy": naive_acc,
            "xgboost_test_accuracy": xgb_acc,
            "accuracy_lift": xgb_acc - naive_acc if pd.notnull(naive_acc) and pd.notnull(xgb_acc) else np.nan,
            "mean_strategy_return": traded["strategy_return"].mean() if len(traded) > 0 else np.nan,
            "hit_rate": (traded["strategy_return"] > 0).mean() if len(traded) > 0 else np.nan,
        })

        importance_df = trainer.feature_importance_table(
            model=model,
            feature_cols=top_features,
            ticker=ticker
        )
        importance_df["split_id"] = split_id
        all_feature_importance.append(importance_df)

        test_conf = pd.Series(final_test_df["predicted_probability"])
        all_probability_summary.append({
            "Ticker": ticker,
            "split_id": split_id,
            "test_prob_q25": float(test_conf.quantile(0.25)),
            "test_prob_median": float(test_conf.median()),
            "test_prob_q75": float(test_conf.quantile(0.75)),
            "test_prob_max": float(test_conf.max()),
            "meta_prob_q25": float(pd.Series(meta_test_prob).quantile(0.25)),
            "meta_prob_median": float(pd.Series(meta_test_prob).median()),
            "meta_prob_q75": float(pd.Series(meta_test_prob).quantile(0.75)),
            "meta_prob_max": float(pd.Series(meta_test_prob).max()),
            "train_start_date": train_df["Date"].min(),
            "train_end_date": train_df["Date"].max(),
            "test_start_date": test_df["Date"].min(),
            "test_end_date": test_df["Date"].max(),
            "train_rows": len(train_df),
            "test_rows": len(test_df),
        })

In [0]:
metrics_df = pd.DataFrame(all_metrics)
baseline_compare_df = pd.DataFrame(all_baseline_compare)
all_predictions_df = pd.concat(all_predictions, axis=0, ignore_index=True)
importance_df = pd.concat(all_feature_importance, axis=0, ignore_index=True)
probability_summary_df = pd.DataFrame(all_probability_summary)
feature_selection_df = pd.DataFrame(all_feature_selection)
threshold_optimization_df = pd.concat(all_threshold_optimization, axis=0, ignore_index=True)

metrics_df.head()

In [0]:
baseline_compare_df.sort_values("accuracy_lift", ascending=False)

In [0]:
probability_summary_df

In [0]:
importance_df.groupby("Ticker").head(10)

In [0]:
# metrics table
spark.createDataFrame(metrics_df.drop('Ticker', axis=1)) \
    .write \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(MODEL_METRICS_TABLE_NAME)

# predictions table
spark.createDataFrame(all_predictions_df) \
    .write \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(MODEL_PREDICTIONS_TABLE_NAME)

# feature importance table
spark.createDataFrame(importance_df) \
    .write \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(FEATURE_IMPORTANCE_TABLE_NAME)

# baseline compare table
spark.createDataFrame(baseline_compare_df) \
    .write \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(BASELINE_COMPARE_TABLE_NAME)

# probability summary table
spark.createDataFrame(probability_summary_df) \
    .write \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("p2_probability_summary")
    
spark.createDataFrame(feature_selection_df) \
    .write \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("p2_feature_selection")

spark.createDataFrame(threshold_optimization_df) \
    .write \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("p2_threshold_optimization")

In [0]:
importance_df = trainer.feature_importance_table(
    model=model,
    feature_cols=top_features,
    ticker=ticker
)
importance_df.head(10)

In [0]:
spark.createDataFrame(metrics_df) \
    .write \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(MODEL_METRICS_TABLE_NAME)

spark.createDataFrame(all_predictions_df) \
    .write \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(MODEL_PREDICTIONS_TABLE_NAME)

spark.createDataFrame(importance_df) \
    .write \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("p2_feature_importance")

spark.createDataFrame(baseline_compare_df) \
    .write \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("p2_baseline_compare")

In [0]:
baseline_compare_df.groupby("Ticker")["accuracy_lift"].mean().sort_values(ascending=False)

In [0]:
probability_summary_df.groupby("Ticker")[["test_prob_q25","test_prob_median","test_prob_q75","test_prob_max"]].mean()

In [0]:
importance_df[importance_df["feature"].str.contains("news|sentiment", case=False, na=False)].head(30)